# Pipeline Logging Demo

This notebook demonstrates the user-facing logging added by #14. It requires a running NDD/DataDesigner backend. If unavailable, expected output is shown in markdown cells below each code cell.

In [ ]:
from anonymizer.logging import configure_logging

configure_logging()  # use configure_logging(verbose=True) to see DataDesigner engine logs

## Configuration

Set `GLINER_ENDPOINT` below to point to your GLiNER (NeMo Retriever NIM) instance. This creates a separate provider for the GLiNER model so it doesn't affect the other models that use the default `nvidia` provider.

In [2]:
# Change this to your GLiNER NIM endpoint URL
GLINER_ENDPOINT = ""

# YAML override: points the gliner-pii-detector model at a custom provider
MODEL_CONFIGS = f"""
model_configs:
  - alias: gliner-pii-detector
    model: nvidia/nemotron-pii
    provider: gliner-nim
    inference_parameters:
      max_parallel_requests: 16
      temperature: 0.0
      top_p: 1.0
      max_tokens: 1024
      timeout: 120

  - alias: gpt-oss-120b
    model: nvdev/openai/gpt-oss-120b
    provider: nvidia
    inference_parameters:
      max_parallel_requests: 16
      max_tokens: 16384
      temperature: 0.3
      top_p: 0.95
      timeout: 300

  - alias: nemotron-30b-thinking
    model: nvidia/nemotron-3-nano-30b-a3b
    provider: nvidia
    inference_parameters:
      max_parallel_requests: 16
      max_tokens: 8192
      temperature: 0.4
      top_p: 1.0
      timeout: 300
"""

In [3]:
import tempfile
from pathlib import Path

import pandas as pd

tmp_dir = tempfile.mkdtemp(prefix="logging_demo_")
csv_path = Path(tmp_dir) / "sample.csv"

pd.DataFrame({
    "text": [
        "Alice Johnson works at Acme Corp in Portland, Oregon.",
        "Contact Bob Smith at bob.smith@example.com or 555-0123.",
        "Dr. Carol Lee treated patient #12345 on 2024-03-15.",
    ]
}).to_csv(csv_path, index=False)

print(f"Sample data saved to {csv_path}")

Sample data saved to /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_3l9qll8k/sample.csv


## Scenario 1: Full run with Redact replacement

In [4]:
from data_designer.config.default_model_settings import get_default_providers
from data_designer.config.models import ModelProvider

from anonymizer.config.anonymizer_config import AnonymizerConfig, AnonymizerInput
from anonymizer.config.replace_strategies import Redact
from anonymizer.interface.anonymizer import Anonymizer

# Keep existing default providers and add the GLiNER NIM provider
providers = get_default_providers() + [ModelProvider(name="gliner-nim", endpoint=GLINER_ENDPOINT)]

anonymizer = Anonymizer(
    model_configs=MODEL_CONFIGS,
    model_providers=providers,
)
result = anonymizer.run(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
)
result.dataframe

[12:07:33] [INFO] 🔧 Anonymizer initialized with 3 model configs
[12:07:33] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_3l9qll8k/sample.csv (column: 'text')
[12:07:33] [INFO] 🔍 Running entity detection on 3 records
[12:07:42] [INFO]   |-- ✅ Detection complete — 12 entities found across 3 records (0 failed)
[12:07:42] [INFO] 🔄 Running Redact replacement
[12:07:42] [INFO]   |-- ✅ Replacement complete (0 failed)
[12:07:42] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"[{'end_position': 5, 'id': 'c57888669b3079df',...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"[{'end_position': 11, 'id': '09d947c68469fb15'...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...
2,Dr. Carol Lee treated patient #12345 on 2024-0...,Dr. <first_name>Carol</first_name> <last_name>...,"[{'end_position': 9, 'id': 'f3700dd6a43b057f',...",Dr. [REDACTED_FIRST_NAME] [REDACTED_LAST_NAME]...


Expected log output:
```
anonymizer — 🔧 Anonymizer initialized with N model configs
anonymizer — 📂 Loaded 3 records from /tmp/.../sample.csv (column: 'text')
anonymizer — 🔍 Running entity detection on 3 records
anonymizer —   |-- ✅ Detection complete — X entities found across 3 records (0 failed)
anonymizer — 🔄 Running Redact replacement
anonymizer —   |-- ✅ Replacement complete (0 failed)
anonymizer — 🎉 Pipeline complete — 3 records processed, 0 total failures
```

## Scenario 2: Preview mode

In [5]:
preview = anonymizer.preview(
    config=AnonymizerConfig(replace=Redact()),
    data=AnonymizerInput(source=str(csv_path)),
    num_records=2,
)
preview.dataframe

[12:07:42] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_3l9qll8k/sample.csv (column: 'text')
[12:07:42] [INFO]   |-- 👀 Preview mode: processing 2 of 3 records
[12:07:42] [INFO] 🔍 Running entity detection on 3 records
[12:07:49] [INFO]   |-- ✅ Detection complete — 9 entities found across 2 records (0 failed)
[12:07:49] [INFO] 🔄 Running Redact replacement
[12:07:49] [INFO]   |-- ✅ Replacement complete (0 failed)
[12:07:49] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities,text_replaced
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"[{'id': 'c57888669b3079df', 'value': 'Alice', ...",[REDACTED_FIRST_NAME] [REDACTED_LAST_NAME] wor...
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"[{'id': '09d947c68469fb15', 'value': 'Bob', 'l...",Contact [REDACTED_FIRST_NAME] [REDACTED_LAST_N...


Expected log output:
```
anonymizer — 📂 Loaded 3 records from /tmp/.../sample.csv (column: 'text')
anonymizer —   |-- 👀 Preview mode: processing 2 of 3 records
anonymizer — 🔍 Running entity detection on 3 records
anonymizer —   |-- ✅ Detection complete — X entities found across 2 records (0 failed)
anonymizer — 🔄 Running Redact replacement
anonymizer —   |-- ✅ Replacement complete (0 failed)
anonymizer — 🎉 Pipeline complete — 3 records processed, 0 total failures
```

## Scenario 3: Detection only (no replacement)

In [6]:
from anonymizer.config.rewrite import PrivacyGoal
from anonymizer.config.anonymizer_config import Rewrite

result_detect_only = anonymizer.run(
    config=AnonymizerConfig(rewrite=Rewrite()),
    data=AnonymizerInput(source=str(csv_path)),
)
result_detect_only.dataframe

[12:07:49] [INFO] 📂 Loaded 3 records from /var/folders/66/lhpfpgl94z33ykf1njyb9y080000gp/T/logging_demo_3l9qll8k/sample.csv (column: 'text')
[12:07:49] [INFO] 🔍 Running entity detection on 3 records
[12:08:06] [INFO]   |-- ✅ Detection complete — 13 entities found across 3 records (0 failed)
[12:08:06] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


,text,text_with_spans,final_entities
0,"Alice Johnson works at Acme Corp in Portland, ...",<first_name>Alice</first_name> <last_name>John...,"[{'end_position': 5, 'id': 'c57888669b3079df',..."
1,Contact Bob Smith at bob.smith@example.com or ...,Contact <first_name>Bob</first_name> <last_nam...,"[{'end_position': 11, 'id': '09d947c68469fb15'..."
2,Dr. Carol Lee treated patient #12345 on 2024-0...,<occupation>Dr.</occupation> <first_name>Carol...,"[{'end_position': 3, 'id': '44fcc37f45ef1be4',..."


Expected log output:
```
anonymizer — 📂 Loaded 3 records from /tmp/.../sample.csv (column: 'text')
anonymizer — 🔍 Running entity detection on 3 records
anonymizer —   |-- ✅ Detection complete — X entities found across 3 records (0 failed)
anonymizer — 🎉 Pipeline complete — 3 records processed, 0 total failures
```
(No replacement log messages appear)

## Cleanup

This notebook is temporary and should be removed before merging.